# Model Test Notebook

This notebook gives one test example for each backend model using the exact input names each model expects.

In [8]:
from types import SimpleNamespace

from Run_Model import run_model as run_distribution_model
from YAC_Model import run_model as run_yac_model
from Pass_Model_JP import execute_pass_model

## Run Model Test

Expected fields for `Run_Model.run_model(play)`:
- `LOS`
- `goal_to_go`
- `down`
- `ydstogo`
- `formation` with values like `"shotgun"`, `"qbdropback"`, or `"both"`
- `run_location` with values `"left"`, `"middle"`, or `"right"`
- `run_gap` with values `"guard"`, `"tackle"`, or `"end"`

In [9]:
run_play = SimpleNamespace(
    LOS=48,
    goal_to_go=0,
    down=2,
    ydstogo=5,
    formation="shotgun",
    run_location="right",
    run_gap="guard",
)

run_result = run_distribution_model(run_play)
run_result

({'Gain/Loss Range': [(-1000000.0, -48),
   (-48, -45),
   (-45, -40),
   (-40, -35),
   (-35, -30),
   (-30, -25),
   (-25, -20),
   (-20, -15),
   (-15, -10),
   (-10, -5),
   (-5, -0.5),
   (-0.5, 0.5),
   (0.5, 5),
   (5, 10),
   (10, 15),
   (15, 20),
   (20, 25),
   (25, 30),
   (30, 35),
   (35, 40),
   (40, 45),
   (45, 50),
   (50, 52),
   (52, 1000000.0)],
  'Yardline Range': [(-10, 0),
   (0, 3),
   (3, 8),
   (8, 13),
   (13, 18),
   (18, 23),
   (23, 28),
   (28, 33),
   (33, 38),
   (38, 43),
   (43, 47.5),
   (47.5, 48.5),
   (48.5, 53),
   (53, 58),
   (58, 63),
   (63, 68),
   (68, 73),
   (73, 78),
   (78, 83),
   (83, 88),
   (88, 93),
   (93, 98),
   (98, 100),
   (100, 110)],
  'Label': ['Safety',
   'Loss of 48 to 45',
   'Loss of 45 to 40',
   'Loss of 40 to 35',
   'Loss of 35 to 30',
   'Loss of 30 to 25',
   'Loss of 25 to 20',
   'Loss of 20 to 15',
   'Loss of 15 to 10',
   'Loss of 10 to 5',
   'Loss of 5 to 0.5',
   'No Gain',
   'Gain of 0.5 to 5',
   'Ga

## Pass -> YAC Flow

The pass model wrapper used here is `execute_pass_model(play)`, which accepts a single play object.

### Variable passed from pass model into YAC model

- `pass_completion`

In this setup:
- `execute_pass_model(play)` returns a dictionary that includes a completion probability
- `YAC_Model.run_model(play)` expects a field named `pass_completion`
- so the pass model's output probability should be assigned to `play.pass_completion`

### Other values that should stay aligned between the two models

These are not outputs of the pass model, but they describe the same pass play and should usually be the same in both calls:
- `down`
- `ydstogo`
- `shotgun`
- `air_yards`
- `pass_location`
- team/season context

## Pass Model Test Using `execute_pass_model(play)`

Expected play-object fields for `execute_pass_model(play)`:
- `season`
- `offense_team`
- `defense_team`
- `down`
- `quarter`
- `shotgun`
- `LOS`
- `ydstogo`
- `pass_attempt_length`
- `pass_location`

In [10]:
pass_play = SimpleNamespace(
    season=2010,
    offense_team="SEA",
    defense_team="DEN",
    down=2,
    quarter=2,
    shotgun=0,
    LOS=48,
    goal_to_go=0,
    ydstogo=5,
    pass_attempt_length=5,
    pass_location="right",
)

pass_result = execute_pass_model(pass_play)
pass_result

{'play_type': 'pass',
 'probability': 0.6351712716054713,
 'probability_percent': 63.52}

## YAC Model Test Using Pass Model Output

Expected fields for `YAC_Model.run_model(play)`:
- `LOS`
- `goal_to_go`
- `down`
- `ydstogo`
- `posteam`
- `defteam`
- `season`
- `pass_location`
- `shotgun`
- `air_yards`
- `pass_completion`

Below, `pass_completion` is set equal to the pass model output.

In [11]:
yac_play = SimpleNamespace(
    LOS=pass_play.LOS,
    goal_to_go=pass_play.goal_to_go,
    down=str(pass_play.down),
    ydstogo=pass_play.ydstogo,
    posteam=pass_play.offense_team,
    defteam=pass_play.defense_team,
    season=pass_play.season,
    pass_location=pass_play.pass_location,
    shotgun=pass_play.shotgun,
    air_yards=pass_play.pass_attempt_length,
    pass_completion=float(pass_result["probability"]),
)

yac_result = run_yac_model(yac_play)
yac_result

({'Gain/Loss Range': [(-1000000.0, -48),
   (-48, -45),
   (-45, -40),
   (-40, -35),
   (-35, -30),
   (-30, -25),
   (-25, -20),
   (-20, -15),
   (-15, -10),
   (-10, -5),
   (-5, -0.5),
   (-0.5, 0.5),
   (0.5, 5),
   (5, 10),
   (10, 15),
   (15, 20),
   (20, 25),
   (25, 30),
   (30, 35),
   (35, 40),
   (40, 45),
   (45, 50),
   (50, 52),
   (52, 1000000.0)],
  'Yardline Range': [(-10, 0),
   (0, 3),
   (3, 8),
   (8, 13),
   (13, 18),
   (18, 23),
   (23, 28),
   (28, 33),
   (33, 38),
   (38, 43),
   (43, 47.5),
   (47.5, 48.5),
   (48.5, 53),
   (53, 58),
   (58, 63),
   (63, 68),
   (68, 73),
   (73, 78),
   (78, 83),
   (83, 88),
   (88, 93),
   (93, 98),
   (98, 100),
   (100, 110)],
  'Label': ['Safety',
   'Loss of 48 to 45',
   'Loss of 45 to 40',
   'Loss of 40 to 35',
   'Loss of 35 to 30',
   'Loss of 30 to 25',
   'Loss of 25 to 20',
   'Loss of 20 to 15',
   'Loss of 15 to 10',
   'Loss of 10 to 5',
   'Loss of 5 to 0.5',
   'No Gain',
   'Gain of 0.5 to 5',
   'Ga